# Chapter 12: Search, Similarity and Geospatial

**Level 3: Integration & Engineering** · ⏱ about 3 hours · Datasets: *Recipes*, *EduGraph lessons*, *Supply chain network*

### What you will learn

1. How to **switch on** database features that are off by default, and why some need the database briefly **offline**
2. **Full-text search** with `textMatch`: scores, wildcards, phrases, fuzzy matching, and why results must be **scoped**
3. **Similarity search**: "recipes similar to X" with a Stardog similarity model, compared with **embedding** similarity
4. **Geospatial queries** with GeoSPARQL: points, distances, "near", "within a region"
5. **Hands-on:** keyword search across recipes and lessons, similar recipes, and a nearest-warehouse finder

This chapter **changes two database settings** (`search.enabled`, `spatial.enabled`) and restores them in the clean-up at the end.

In [1]:
import sys
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import stardog
from rdflib import Literal, URIRef

from kg import client, dboptions, extraction, sparql
from kg.recipes import RECIPE

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)
conn = client.connect()
PREFIX = 'urn:tutorial:ch12'
ROOT = Path('..')
RECIPES, LESSONS, NETWORK = f'{PREFIX}:recipes', f'{PREFIX}:lessons', f'{PREFIX}:network'

## 12.1 Switching features on

Full-text and geospatial search keep **extra indexes** next to the triples, so they're off by default. Turning them on is a database **option** (Chapter 2). Both options can only change while the database is **offline**, because Stardog must set up the new indexes before anyone queries.

`kg.dboptions.set_options()` tries online first, then briefly takes the database offline if needed, and always brings it back. It returns the previous values, so we can restore them later:

In [2]:
print(Path(dboptions.__file__).read_text())

"""Read and change database options (Chapter 12+).

Some options (e.g. search.enabled, spatial.enabled) can only change while the database is
offline. set_options() tries online first, then briefly takes the database offline, and always
brings it back online.
"""
import stardog

from kg import client


def get_options(*names: str) -> dict:
  with client.admin() as admin:
    return admin.database(client.load_settings().database).get_options(*names)


def set_options(options: dict) -> dict:
  """Apply `options`; returns their previous values so they can be restored later."""
  with client.admin() as admin:
    db = admin.database(client.load_settings().database)
    previous = db.get_options(*options)
    changes = {k: v for k, v in options.items() if previous.get(k) != v}
    if not changes:
      return previous
    try:
      db.set_options(changes)
    except stardog.exceptions.StardogException as e:
      if 'while the database' not in str(e):
        raise
      db.offline()      

In [3]:
ORIGINAL = dboptions.set_options({'search.enabled': True, 'spatial.enabled': True})
print('before:', ORIGINAL)
print('now:   ', dboptions.get_options('search.enabled', 'spatial.enabled'))

before: {'spatial.enabled': False, 'search.enabled': False}


now:    {'spatial.enabled': True, 'search.enabled': True}


> ⚠️ **Offline means offline for everyone.** On a shared or production database, schedule option changes like any other maintenance. On this tutorial database it takes about a second.

Now load the data. Loading **after** switching the features on means everything is indexed as it arrives:

In [4]:
lesson_ttl = ['@prefix edu: <http://example.org/edu#> .']
for f in sorted((ROOT / 'data' / 'edugraph' / 'lessons').glob('*.md')):
    text = f.read_text(encoding='utf-8')
    lesson_ttl.append(f'<http://example.org/edu/document/{f.stem}> a edu:Document ; '
                      f'edu:title {sparql.term(text.splitlines()[0][2:])} ; edu:text {sparql.term(text)} .')

with client.transaction(conn) as stats:
    for g in (RECIPES, LESSONS, NETWORK):
        conn.clear(graph_uri=g)
    conn.add(stardog.content.File(str(ROOT / 'data' / 'recipes' / 'recipes.ttl')), graph_uri=RECIPES)
    conn.add(stardog.content.Raw('\n'.join(lesson_ttl).encode(), 'text/turtle'), graph_uri=LESSONS)
    conn.add(stardog.content.File(str(ROOT / 'data' / 'supply_chain' / 'network.ttl')), graph_uri=NETWORK)
print('Loaded:', stats)

Loaded: {'added': 1536, 'removed': 0}


The six lesson texts from Chapter 11 are stored in full as `edu:text`, so we can search inside them.

## 12.2 Full-text search

`FILTER(CONTAINS(…))` (Chapter 4) scans every string and knows nothing about words. A **full-text index** (Stardog uses Lucene) splits text into words, so it answers keyword searches fast, **ranks** results by relevance, and supports wildcards, phrases and fuzzy matching.

The query form is a `SERVICE` block:

```sparql
SERVICE fts:textMatch {
  [] fts:query "chicken" ;        # the search expression
     fts:result ?text ;           # each matching literal
     fts:score ?score .           # its relevance
}
```

In [5]:
FTS = '''PREFIX fts: <tag:stardog:api:search:>
PREFIX rec: <http://example.org/recipes#>
PREFIX edu: <http://example.org/edu#>
'''
def search_literals(query, limit=8):
    '''Raw full-text search: matching literals and scores, from the WHOLE database.'''
    return sparql.run_query(conn, FTS + '''SELECT ?text ?score WHERE {
      SERVICE fts:textMatch { [] fts:query ?q ; fts:result ?text ; fts:score ?score . } }
      ORDER BY DESC(?score)''', limit=limit, q=query)

search_literals('chicken')

,text,score
0,"Preheat oven to 375ÔøΩF (190ÔøΩC). Dip chicken breasts in beaten eggs, then ...",2.771589
1,Season chicken breasts with salt and pepper. Heat oil in a skillet and cook ...,2.715972
2,Preheat oven to 375ÔøΩF (190ÔøΩC). Flatten chicken breasts to 1/4-inch thick...,2.715972
3,"In a large pot, heat oil over medium heat. Add diced onions, carrots, and ce...",2.468316
4,"Dice ripe mangoes, tomatoes, red onions, and jalapeÔøΩos. Chop cilantro and ...",2.087601
5,Chicken Caesar Salad,2.087601
6,Chicken Noodle Soup,2.087601
7,Chicken Parmesan,2.087601


### Scope your search

Look carefully: some hits may end in *"Serves 3, 25 min."*. Those are **not** from this chapter's graphs. The full-text index covers the **whole database**, and graph parameters such as `default_graph_uri` don't restrict it. To search only your data, **join** each hit back to a triple in your graphs:

In [6]:
def search_recipes(query):
    '''Recipes whose name or description matches, ranked by score (only this chapter's recipes).'''
    return sparql.run_query(conn, FTS + '''
    SELECT ?recipe (MAX(?score) AS ?best) (SAMPLE(?text) AS ?matched)
    WHERE {
      SERVICE fts:textMatch { [] fts:query ?q ; fts:result ?text ; fts:score ?score . }
      ?r a rec:Recipe ; rec:name|rec:description ?text ; rec:name ?recipe .     # the join scopes it
      FILTER(lang(?recipe) = "en")
    }
    GROUP BY ?recipe ORDER BY DESC(?best)''', graphs=RECIPES, q=query)

search_recipes('chicken')

,recipe,best,matched
0,Crispy Chicken Tenders,2.087601,Crispy Chicken Tenders
1,Thai Green Curry,2.087601,"Chicken simmered in coconut milk and green curry paste, served with rice."
2,Chicken Stir Fry,2.087601,"Chicken and vegetables stir-fried in soy sauce and sesame oil, served with r..."


Because the matched text must be a `rec:name` or `rec:description` of a `rec:Recipe` **in `RECIPES`**, text from elsewhere drops out.

### Search syntax

The query string uses Lucene syntax:

In [7]:
for q in ['rice', 'chick*', '"coconut milk"', 'garlic AND cream', 'salman~', 'noodles OR pasta']:
    hits = search_recipes(q)
    print(f'{q:20} {len(hits):2} hits: {", ".join(hits.recipe)}')

rice                  3 hits: Pad Thai, Thai Green Curry, Chicken Stir Fry


chick*                3 hits: Crispy Chicken Tenders, Thai Green Curry, Chicken Stir Fry


"coconut milk"        1 hits: Thai Green Curry


garlic AND cream      1 hits: Moules Marinieres


salman~               2 hits: Grilled Salmon, Potato Salad


noodles OR pasta      3 hits: Spaghetti Carbonara, Pesto Pasta, Pad Thai


| Syntax | Meaning |
|---|---|
| `rice` | contains the word *rice* (case-insensitive, whole words) |
| `chick*` | wildcard: *chicken*, *chickpea*, … |
| `"coconut milk"` | exact phrase |
| `garlic AND cream`, `noodles OR pasta` | boolean operators |
| `salman~` | fuzzy: tolerates typos (*salmon*), and here also catches *salad*, which is one letter away too |

### Search plus structure

The real strength is **combining** keyword search with graph patterns. *Quick recipes that mention rice*:

In [8]:
sparql.run_query(conn, FTS + '''
SELECT DISTINCT ?recipe ?minutes WHERE {
  SERVICE fts:textMatch { [] fts:query "rice" ; fts:result ?text . }
  ?r rec:description ?text ; rec:name ?recipe ; rec:prepMinutes ?minutes .
  FILTER(lang(?recipe) = "en" && ?minutes <= 30)
} ORDER BY ?minutes''', graphs=RECIPES)

,recipe,minutes
0,Pad Thai,25
1,Chicken Stir Fry,25


### Hands-on: find resources by keyword

One search box across **recipes and lessons**: search every indexed literal, then map each hit to the resource it belongs to:

In [9]:
def find(query, limit=8):
    return sparql.run_query(conn, FTS + '''
    SELECT ?kind ?title (ROUND(MAX(?score) * 100) / 100 AS ?relevance)
    WHERE {
      SERVICE fts:textMatch { [] fts:query ?q ; fts:result ?text ; fts:score ?score . }
      { ?s a rec:Recipe ; rec:name|rec:description ?text ; rec:name ?title  FILTER(lang(?title) = "en")  BIND("recipe" AS ?kind) }
      UNION
      { ?s a edu:Document ; edu:text|edu:title ?text ; edu:title ?title  BIND("lesson" AS ?kind) }
    }
    GROUP BY ?kind ?title ORDER BY DESC(?relevance)''', graphs=[RECIPES, LESSONS], limit=limit, q=query)

for q in ['regression', 'rice OR pandas', 'join*']:
    print(f'--- {q}')
    display(find(q))

--- regression


,kind,title,relevance
0,lesson,Understanding linear regression,4.29
1,lesson,Evaluating classifiers,3.40


--- rice OR pandas


,kind,title,relevance
0,lesson,Plotting with matplotlib,3.82
1,lesson,Getting started with pandas DataFrames,3.82
2,lesson,Understanding linear regression,3.23
3,recipe,Thai Green Curry,2.25
4,recipe,Pad Thai,2.25
5,recipe,Chicken Stir Fry,2.25


--- join*


,kind,title,relevance
0,lesson,Getting started with pandas DataFrames,1.0
1,lesson,From SQL joins to SPARQL,1.0


> 🔁 **Neo4j equivalent:** Neo4j full-text indexes (`CREATE FULLTEXT INDEX …` and `db.index.fulltext.queryNodes`) are also Lucene-based. There you create an index per label and property; Stardog indexes all string literals of the database.

## 12.3 Similarity search: "recipes similar to X"

Full-text search matches **words**. Similarity search compares **things**: two recipes are similar when they share many ingredients.

### A Stardog similarity model

Stardog trains **similarity models** from SPARQL, and stores them in the special graph `spa:model`. The recipe:

1. **train**: an `INSERT` that gives the model each recipe (`spa:predict`) with its **features**, here its set of ingredients (`spa:set`);
2. **query**: give it the features of a recipe, get the most similar recipes with a **confidence**.

In [10]:
SPA = '''PREFIX spa: <tag:stardog:api:analytics:>
PREFIX rec: <http://example.org/recipes#>
PREFIX recipe: <http://example.org/recipes/recipe/>
'''
MODEL = '<urn:tutorial:ch12:similar-recipes>'

def train_similarity_model():
    conn.update(SPA + f'DELETE WHERE {{ GRAPH spa:model {{ {MODEL} ?p ?o }} }}')      # retrain from scratch
    conn.update(SPA + f'''
    INSERT {{ GRAPH spa:model {{ {MODEL} a spa:SimilarityModel ; spa:arguments (?ingredients) ; spa:predict ?recipe . }} }}
    WHERE  {{ SELECT (spa:set(?i) AS ?ingredients) ?recipe
              WHERE {{ GRAPH <{RECIPES}> {{ ?recipe rec:hasIngredient ?i }} }}
              GROUP BY ?recipe }}''')

def similar_recipes(recipe_iri, n=5):
    x = sparql.term(URIRef(recipe_iri))          # a safely escaped IRI (bindings don't reach into this subquery)
    return sparql.run_query(conn, SPA + f'''
    SELECT ?similar (ROUND(?confidence * 100) / 100 AS ?similarity) WHERE {{
      GRAPH spa:model {{ {MODEL} spa:arguments (?ingredients) ; spa:confidence ?confidence ;
                                 spa:parameters [ spa:limit {n + 1} ] ; spa:predict ?r . }}
      {{ SELECT (spa:set(?i) AS ?ingredients) WHERE {{ GRAPH <{RECIPES}> {{ {x} rec:hasIngredient ?i }} }} }}
      GRAPH <{RECIPES}> {{ ?r rec:name ?similar  FILTER(lang(?similar) = "en") }}
      FILTER(?r != {x})
    }} ORDER BY DESC(?confidence)''')

train_similarity_model()
for name in ['pad-thai', 'pancakes', 'pesto-pasta']:
    print(f'--- similar to {name}')
    print(similar_recipes(RECIPE[name]).to_string(index=False))

--- similar to pad-thai


               similar  similarity
   Spaghetti Carbonara        0.29
      Thai Green Curry        0.27
        Grilled Salmon        0.16
Crispy Chicken Tenders        0.16
           Pesto Pasta        0.14
--- similar to pancakes


               similar  similarity
    Chocolate Brownies        0.67
Crispy Chicken Tenders        0.55
      Margherita Pizza        0.33
   Spaghetti Carbonara        0.33
       Lupin Flatbread        0.24
--- similar to pesto-pasta


            similar  similarity
Spaghetti Carbonara        0.50
     Grilled Salmon        0.37
   Margherita Pizza        0.33
    Lupin Flatbread        0.24
  Moules Marinieres        0.17


Pancakes and Chocolate Brownies share flour, butter, sugar and eggs; Pesto Pasta and Carbonara share spaghetti, Parmesan and garlic. The confidence is a similarity score between ingredient sets (1.0 means identical).

- The model is a **snapshot**: after recipes change, retrain it (`train_similarity_model()`).
- The features can be anything a query can compute: categories, allergens, cuisine, ratings. Choose them to match what "similar" should mean for users.

### A different kind of similar: meaning

Ingredient overlap misses dishes that are *alike* in character but share few ingredients. **Embeddings** of the descriptions (Chapter 11) capture **meaning** instead. If Ollama is running:

In [11]:
if not extraction.ollama_available():
    print('Ollama is not running: skipping the embedding comparison (see Chapter 11 for setup).')
else:
    texts = sparql.run_query(conn, '''PREFIX rec: <http://example.org/recipes#>
        SELECT ?r ?name ?description { ?r rec:name ?name ; rec:description ?description  FILTER(lang(?name) = "en") }''',
        graphs=RECIPES)
    vectors = dict(zip(texts.r, extraction.embed((texts.name + '. ' + texts.description).tolist(), kind='document')))
    names = dict(zip(texts.r, texts.name))

    def similar_by_meaning(recipe_iri, n=5):
        scores = [(extraction.cosine(vectors[str(recipe_iri)], v), names[r]) for r, v in vectors.items() if r != str(recipe_iri)]
        return pd.DataFrame(sorted(scores, reverse=True)[:n], columns=['similarity', 'similar']).round(2)[['similar', 'similarity']]

    for name in ['pad-thai', 'pancakes']:
        print(f'--- similar in meaning to {name}')
        print(similar_by_meaning(RECIPE[name]).to_string(index=False))

--- similar in meaning to pad-thai
         similar  similarity
Chicken Stir Fry        0.82
Thai Green Curry        0.79
    Potato Salad        0.70
  Vegetable Soup        0.69
     Pesto Pasta        0.66
--- similar in meaning to pancakes
               similar  similarity
    Chocolate Brownies        0.70
       Lupin Flatbread        0.69
Crispy Chicken Tenders        0.68
              Pad Thai        0.65
   Spaghetti Carbonara        0.64


| | Stardog similarity model | Embeddings |
|---|---|---|
| "Similar" means | Shares **features** you choose (ingredients) | Similar **meaning** of the text |
| Explainable | Yes: the shared ingredients | Hardly |
| Runs | Inside Stardog, from SPARQL | In Python, with an embedding model |
| Good for | "Other recipes with what I have in the fridge" | "Something like this, in spirit" |

Chapter 16 uses embeddings again, to find the right part of the graph for a question asked in natural language.

## 12.4 Geospatial queries

GeoSPARQL is the W3C standard for location data in RDF. A thing **has a geometry**, and the geometry is written as **WKT** (Well-Known Text):

```turtle
sc:P4 geo:hasGeometry sc:P4-location .
sc:P4-location geo:asWKT "POINT(4.4777 51.9244)"^^geo:wktLiteral .     # Port of Rotterdam
```

> ⚠️ WKT points are written **`POINT(longitude latitude)`**, x before y. Swapping them is the classic mistake: Rotterdam would land in Somalia.

Our supply chain network (from the `supply_chain/` project) now has city coordinates for all 30 facilities.

In [12]:
GEO = '''PREFIX geo:  <http://www.opengis.net/ont/geosparql#>
PREFIX geof: <http://www.opengis.net/def/function/geosparql/>
PREFIX uom:  <http://www.opengis.net/def/uom/OGC/1.0/>
PREFIX sc:   <http://example.org/supplychain#>
'''
def g(query, **params):
    return sparql.run_query(conn, GEO + query, graphs=NETWORK, **params)

g('SELECT ?facility ?city ?wkt { ?f sc:name ?facility ; sc:city ?city ; geo:hasGeometry/geo:asWKT ?wkt } ORDER BY ?city LIMIT 5')

,facility,city,wkt
0,Amsterdam Store,Amsterdam,POINT(4.9041 52.3676)
1,Bangalore Chip Works,Bangalore,POINT(77.5946 12.9716)
2,Minas Gerais Coffee Farm,Belo Horizonte,POINT(-43.9378 -19.9167)
3,Berlin Store,Berlin,POINT(13.405 52.52)
4,Buenos Aires Store,Buenos Aires,POINT(-58.3816 -34.6037)


### Distance

`geof:distance(a, b, unit)` gives the great-circle distance. How long is each sea route?

In [13]:
g('''SELECT ?from ?to ?days (ROUND(geof:distance(?a, ?b, uom:metre) / 1000) AS ?km) (ROUND(?km_raw / ?days) AS ?km_per_day)
WHERE {
  ?r a sc:Route ; sc:mode "Sea" ; sc:leadTimeDays ?days ; sc:origin ?o ; sc:destination ?d .
  ?o sc:name ?from ; geo:hasGeometry/geo:asWKT ?a .
  ?d sc:name ?to   ; geo:hasGeometry/geo:asWKT ?b .
  BIND(geof:distance(?a, ?b, uom:metre) / 1000 AS ?km_raw)
}
ORDER BY DESC(?km)''')

,from,to,days,km,km_per_day
0,Port of Singapore,Port of Los Angeles,20,14121.0,706.0
1,Port of Mumbai,Port of Los Angeles,28,13991.0,500.0
2,Memphis Returns Center,Pune Assembly Plant,30,13810.0,460.0
3,Port of Singapore,Port of Rotterdam,25,10527.0,421.0
4,Port of Shanghai,Port of Los Angeles,18,10434.0,580.0
5,Port of Shanghai,Port of Rotterdam,30,8927.0,298.0
6,Port of Mumbai,Port of Rotterdam,22,6877.0,313.0
7,Port of Shanghai,Port of Singapore,5,3806.0,761.0
8,Shenzhen Assembly Plant,Port of Singapore,5,2603.0,521.0
9,Taipei Display Panels,Shenzhen Assembly Plant,4,812.0,203.0


Straight-line distances ignore canals and coastlines, but they're a good sanity check. A route with an implausible speed (km per day) is a data-quality flag.

### Nearest: which distribution center should serve each store?

For every store, the distance to every distribution center, keeping the closest. That's "best per group", solved with a subquery as in Chapter 6:

In [14]:
g('''SELECT ?store ?nearest_dc ?km
WHERE {
  { SELECT ?s (MIN(?d) AS ?min_d) WHERE {
      ?s a sc:RetailStore ; geo:hasGeometry/geo:asWKT ?sw .
      ?dc a sc:DistributionCenter ; geo:hasGeometry/geo:asWKT ?dw .
      BIND(geof:distance(?sw, ?dw, uom:metre) AS ?d) } GROUP BY ?s }
  ?s sc:name ?store ; geo:hasGeometry/geo:asWKT ?sw .
  ?dc a sc:DistributionCenter ; sc:name ?nearest_dc ; geo:hasGeometry/geo:asWKT ?dw .
  FILTER(ABS(geof:distance(?sw, ?dw, uom:metre) - ?min_d) < 1)
  BIND(ROUND(?min_d / 1000) AS ?km)
}
ORDER BY ?km''')

,store,nearest_dc,km
0,Los Angeles Store,Los Angeles Distribution Center,0.0
1,Chicago Store,Chicago Distribution Center,0.0
2,Amsterdam Store,Rotterdam Distribution Center,57.0
3,Rio de Janeiro Store,Sao Paulo Distribution Center,361.0
4,Houston Store,Dallas Distribution Center,362.0
5,Paris Store,Rotterdam Distribution Center,373.0
6,Berlin Store,Frankfurt Distribution Center,424.0
7,San Francisco Store,Los Angeles Distribution Center,559.0
8,New York Store,Chicago Distribution Center,1144.0
9,Buenos Aires Store,Sao Paulo Distribution Center,1675.0


Compare with the network. Does each store actually receive goods from its nearest DC? Mismatches are candidates for re-routing:

In [15]:
g('''SELECT ?store ?served_by ?nearest
WHERE {
  { SELECT ?s (MIN(?d) AS ?min_d) WHERE {
      ?s a sc:RetailStore ; geo:hasGeometry/geo:asWKT ?sw .
      ?dc a sc:DistributionCenter ; geo:hasGeometry/geo:asWKT ?dw .
      BIND(geof:distance(?sw, ?dw, uom:metre) AS ?d) } GROUP BY ?s }
  ?s sc:name ?store ; geo:hasGeometry/geo:asWKT ?sw .
  ?near a sc:DistributionCenter ; sc:name ?nearest ; geo:hasGeometry/geo:asWKT ?nw .
  FILTER(ABS(geof:distance(?sw, ?nw, uom:metre) - ?min_d) < 1)
  ?supplier a sc:DistributionCenter ; sc:shipsTo ?s ; sc:name ?served_by .
  FILTER(?supplier != ?near)
  FILTER NOT EXISTS { ?near sc:shipsTo ?s }
}
ORDER BY ?store''')

,store,served_by,nearest


**No rows**: every store already receives goods from its nearest distribution center. On a real network this query becomes a standing check, and its rows are candidates for re-routing.

### Near a point

`geof:nearby(point, distance, unit)` returns the **geometries** within a radius. It uses the spatial index, so it doesn't compute every distance. Everything within 1,000 km of the Port of Rotterdam:

In [16]:
g('''SELECT ?facility ?type (ROUND(geof:distance(?w, "POINT(4.4777 51.9244)"^^geo:wktLiteral, uom:metre) / 1000) AS ?km)
WHERE {
  ?geometry geof:nearby ("POINT(4.4777 51.9244)"^^geo:wktLiteral 1000000 uom:metre) .
  ?f geo:hasGeometry ?geometry ; sc:name ?facility ; a ?t .
  ?geometry geo:asWKT ?w .
  BIND(STRAFTER(STR(?t), "#") AS ?type)
}
ORDER BY ?km''')

,facility,type,km
0,Port of Rotterdam,Port,0.0
1,Rotterdam Distribution Center,DistributionCenter,0.0
2,Amsterdam Store,RetailStore,57.0
3,Frankfurt Distribution Center,DistributionCenter,357.0
4,Paris Store,RetailStore,373.0
5,Berlin Store,RetailStore,611.0


### Within a region

`geof:within` finds geometries inside another geometry, such as a **polygon** drawn around a region. Like `nearby`, Stardog evaluates it as a **property function** on the spatial index: `?geometry geof:within <polygon>`. (Written as a `FILTER(geof:within(?a, ?b))` function, it returned no rows in our tests on this Stardog version.) A rough box around South America, with the polygon's corners closed by repeating the first:

In [17]:
SOUTH_AMERICA = Literal('POLYGON((-82 -56, -34 -56, -34 13, -82 13, -82 -56))',
                        datatype=URIRef('http://www.opengis.net/ont/geosparql#wktLiteral'))
g(f'''SELECT ?facility ?city WHERE {{
  ?geometry geof:within {sparql.term(SOUTH_AMERICA)} .
  ?f geo:hasGeometry ?geometry ; sc:name ?facility ; sc:city ?city .
}} ORDER BY ?city''')

,facility,city
0,Minas Gerais Coffee Farm,Belo Horizonte
1,Buenos Aires Store,Buenos Aires
2,Rio de Janeiro Store,Rio de Janeiro
3,Santos Coffee Roastery,Santos
4,Sao Paulo Distribution Center,Sao Paulo


The polygon is a typed `geo:wktLiteral`, written into the query with `sparql.term()` so it's correctly escaped. The same query works for any shape.

> 🔁 **Neo4j equivalent:** Neo4j has a `point` type and `point.distance()`, plus `point.withinBBox()` for boxes. Polygons and the full GeoSPARQL function set need a plugin.

## 12.5 Exercises

**Exercise 1.** Search recipes for **"tomato OR basil"**. Then change `search_recipes` so it also returns the **French name** of each hit.

<details><summary>Solution</summary>

```python
sparql.run_query(conn, FTS + '''SELECT ?recipe ?nom (MAX(?score) AS ?best) WHERE {
  SERVICE fts:textMatch { [] fts:query ?q ; fts:result ?text ; fts:score ?score . }
  ?r rec:name|rec:description ?text ; rec:name ?recipe , ?nom .
  FILTER(lang(?recipe) = "en" && lang(?nom) = "fr") }
GROUP BY ?recipe ?nom ORDER BY DESC(?best)''', graphs=RECIPES, q='tomato OR basil')
```
</details>

**Exercise 2.** Train a second similarity model where a recipe's features are its **allergens** instead of its ingredients, and find recipes with an allergen profile similar to Pad Thai's.

<details><summary>Solution</summary>

Copy `train_similarity_model` and `similar_recipes` with another model IRI, and replace `?recipe rec:hasIngredient ?i` by `?recipe rec:hasIngredient/rec:hasAllergen ?i` (in both the training and the query).
</details>

**Exercise 3.** Which **ports** are within 2,000 km of **Singapore**?

<details><summary>Solution</summary>

```python
g('''SELECT ?port WHERE {
  ?geometry geof:nearby ("POINT(103.8198 1.3521)"^^geo:wktLiteral 2000000 uom:metre) .
  ?p a sc:Port ; sc:name ?port ; geo:hasGeometry ?geometry }''')
```
</details>

**Exercise 4.** Full-text search returned text from outside this chapter's graphs. Why is that a **privacy** concern, and how does the join fix it?

<details><summary>Solution</summary>

The index covers every string literal in the database. A search box that queries the index directly could show users text from graphs they should never see (other projects, other customers). Joining each hit to a pattern in the permitted graphs only returns text that's actually in them. Chapter 14 adds named-graph **permissions**, which enforce this on the server.
</details>

In [18]:
# your answers here

## 12.6 Clean up

Removes this chapter's graphs and the similarity model, and **restores** the two database options to their values before the chapter (the database goes offline for a moment again).

In [19]:
CLEAN_UP = True

if CLEAN_UP:
    conn.update(SPA + f'DELETE WHERE {{ GRAPH spa:model {{ {MODEL} ?p ?o }} }}')
    with client.transaction(conn):
        for graph in (RECIPES, LESSONS, NETWORK):
            conn.clear(graph_uri=graph)
    dboptions.set_options(ORIGINAL)
    print('Removed graphs and model; options now:', dboptions.get_options('search.enabled', 'spatial.enabled'))
conn.close()

Removed graphs and model; options now: {'spatial.enabled': False, 'search.enabled': False}


## Summary

- Some features need **database options**; `search.enabled` and `spatial.enabled` can only change **offline** (`kg.dboptions.set_options` handles that and returns the old values).
- **Full-text search** (`SERVICE fts:textMatch`) finds and **ranks** literals by keywords, with wildcards, phrases, boolean operators and fuzzy matching. The index is **database-wide**: always **scope** hits by joining them to your graphs.
- **Similarity models** (`spa:`) find similar things by shared **features** you choose, trained and queried in SPARQL. They're a snapshot, so retrain after changes. **Embeddings** compare **meaning** instead.
- **GeoSPARQL**: `geo:hasGeometry` / `geo:asWKT` with `POINT(lon lat)`; `geof:distance`, `geof:nearby` (index-backed radius search) and `geof:within` (inside a polygon) combine freely with ordinary graph patterns.

**Next: Chapter 13, GraphQL and building APIs.** Serve the EduGraph through a FastAPI REST service backed by stored SPARQL queries, including `/students/{id}/gaps`, and query the same data with Stardog's GraphQL.